# Stochastik programmieren

Wir wollen uns nun kurz damit beschäftigen, wie sich mit Python Stichproben (en. Samples) von Zufallsexperimenten simulieren lassen.

Verwendete Python-Konstrukte: [Module/import](https://docs.python.org/3/tutorial/modules.html), [Methoden/def](https://docs.python.org/3/tutorial/controlflow.html#defining-functions), [Länge/len](https://docs.python.org/3/library/functions.html#len), [Mengen/set](https://docs.python.org/3/tutorial/datastructures.html#sets), [Generator-Ausdruck](https://docs.python.org/3/tutorial/classes.html#generator-expressions), [Range](https://docs.python.org/3/tutorial/controlflow.html#the-range-function), [Assertion/assert](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement)

## Wahrscheinlichkeitsmaß implementieren

In [1]:
from fractions import Fraction

def P(A, Omega):
    """Die Wahrscheinlichkeit für das Ereignis A,
       gegeben gleich wahrscheinliche Ergebnisse aus einem Ergebnisraum Ω."""
    return Fraction(len(A & Omega), len(Omega))

Wir werden damit nun einen Würfelwurf programmieren:

In [2]:
W = {1, 2, 3, 4, 5, 6}
gerade = set((x*2 for x in range(1,4)))
print("Gerade Würfelaugen:",gerade)

print("Wahrscheinlichkeit für gerade Augenzahl:", P(gerade, W))

Gerade Würfelaugen: {2, 4, 6}
Wahrscheinlichkeit für gerade Augenzahl: 1/2


Damit haben wir nun das Wahrscheinlichkeitsmaß. Wenn wir eine Stichprobe ziehen wollen, müssen wir noch irgendwo den "Zufall" her bekommen.

## Zufall importieren

Python bietet mit dem `random`-Modul eine Schnittstelle zu Pseudozufallszahlen. Die Methode `random.random` ist ein direkt in C implementierter [Mersenne Twister](https://de.wikipedia.org/wiki/Mersenne-Twister). Wenn man "echte" Zufallszahlen braucht, etwa für kryptografische Zwecke, gibt es dazu das `secrets`-Modul. Den Seed für den Mersenne Twister kann man angeben, und sollte man auch, um Zufallssimulationen reproduzierbar zu machen.

In [3]:
from random import random as r
help(r)
print("1", r())
print("2", r())

import random
random.seed(1)
very_random = r()
print("3", very_random)
assert very_random == 0.13436424411240122
print("4", r())
random.seed(1)
print("5", r())

Help on built-in function random:

random() method of random.Random instance
    random() -> x in the interval [0, 1).

1 0.43348847526154644
2 0.171989584157332
3 0.13436424411240122
4 0.8474337369372327
5 0.13436424411240122


![XKCD 221: Random Number](images/random_number.png "RFC 1149.5 specifies 4 as the standard IEEE-vetted random number.")

[Link zum Comic (Randall Munroe, CC-BY-NC 2.5)](https://xkcd.com/221)

Aus einer (Pseudo)zufallszahl zwischen $0$ und $1$ (man beachte: evtl. $0$ aber nie $1$) lassen sich zufällige Würfelwürfe erzeugen:

In [4]:
from math import floor
def transform_unit_to_dice(x):
    return floor(1 + 6*x)

assert list(range(1,7)) == [transform_unit_to_dice((x-1)/6)
                            for x in range(1,7)]

N = 100
würfe = [transform_unit_to_dice(r()) for n in range(N)]
print(würfe)

[6, 5, 2, 3, 3, 4, 5, 1, 1, 6, 3, 5, 1, 3, 5, 2, 6, 6, 1, 1, 4, 6, 3, 2, 3, 1, 2, 3, 3, 2, 2, 2, 3, 2, 1, 6, 4, 4, 2, 6, 6, 1, 2, 5, 5, 6, 3, 5, 5, 2, 4, 6, 6, 4, 4, 1, 2, 5, 3, 2, 4, 5, 5, 3, 3, 4, 5, 4, 3, 3, 1, 1, 5, 6, 4, 3, 2, 4, 6, 5, 4, 6, 2, 4, 6, 4, 3, 2, 4, 6, 1, 5, 5, 6, 5, 5, 4, 4, 3, 1]


Damit man solche Transformationen nicht andauernd programmieren muss, kann man hier auch auf `random.randint(1,6)` oder auch auf `random.choice(range(1,7))` oder `random.randrange(1,7)` zurückgreifen. Dabei ist `randint` ein Kürzel für das entsprechende `randrange` und `choice` ist etwas allgemeiner.

Wir wollen aber festhalten: gegeben eine gleichverteilte "Zufallsvariable" $X=$`random.random` mit Werten in $[0,1)$ haben wir eine Abbildung $t =$`transform_unit_to_dice` konstruiert und implementiert, die Werte in $\Omega := \{1,2,3,4,5,6\}$ hat und $t(X)$ ist gleichverteilt. Die mathematische Abbildung $t \colon [0,1) \to \Omega$ ist eine Zufallsvariable, wir behandeln die [Verknüpfung](https://de.wikipedia.org/wiki/Komposition_(Mathematik)) $t \circ X$ als Zufallsvariable, die den Würfel modelliert.

Nun könnte man sich beschweren: `random.random()` nimmt gar keinen Parameter, ist also keine mathematische Abbildung von einem Definitionsbereich in die Menge $[0,1)$. Tatsächlich müssen wir uns vorstellen, dass es eine Abbildung $X \colon \Omega' \to [0,1)$ ist, und auf $\Omega'$ ein irgendwie geartetes Wahrscheinlichkeitsmaß definiert ist, sodass durch $X$ auf $[0,1)$ die Gleichverteilung induziert wird. Die Menge $\Omega'$ spielt für uns keine konkrete Rolle - da "kommt der Zufall her" und in der Notation `random.random()` sehen wir schon, dass wir eben kein konkretes Element von $\Omega'$ einsetzen, sondern pseudozufällig eins ziehen und das in $X$ einsetzen.

## Größere Stichproben

Die Methode `random.sample(population, k)` erlaubt es eine Stichprobe der Größe $k$ aus einer Population (einer Urne) zu ziehen - ziehen **ohne** Zurücklegen, die gezogenen Elemente sind also paarweise verschieden. Für $k=1$ entspricht das einer Gleichverteilung auf der Population.
Mit der Methode `random.choices(population, weights=None, *, cum_weights=None, k=1)` kann man $k$-mal **mit** Zurücklegen aus der Population ziehen (und dabei anstelle einer geeigneten Population auch Gewichte vergeben).

## Ausblick

Die relativen Häufigkeiten, die wir in diesem Notebook für die Augenzahlen $1,\dots,6$ gezählt haben, sind Schätzungen der Wahrscheinlichkeiten $p(1),\dots,p(6)$ des Würfels. Im nächsten Kapitel nennen wir die Funktion $p$ die *Wahrscheinlichkeitsmassefunktion* (pmf) der Zufallsvariable 'Augenzahl' und machen aus dem Würfelwurf ein erstes statistisches Modell.